In [1]:
!pip -q install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 7.4 MB/s eta 0:0

In [2]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor

In [7]:
!unzip /content/super-ai-engineer-ss-6-heart-disease-prediction.zip

Archive:  /content/super-ai-engineer-ss-6-heart-disease-prediction.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [8]:
train_data = pd.read_csv('/content/train.csv')
test_data = pd.read_csv('/content/test.csv')

In [9]:
train_data.columns

Index(['ID', 'History of HeartDisease or Attack', 'High Blood Pressure',
       'Told High Cholesterol', 'Cholesterol Checked', 'Body Mass Index',
       'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes',
       'Leisure Physical Activity', 'Heavy Alcohol Consumption',
       'Health Care Coverage', 'Doctor Visit Cost Barrier', 'General Health',
       'Difficulty Walking', 'Sex', 'Education Level', 'Income Level', 'Age',
       'Vegetable or Fruit Intake (1+ per Day)'],
      dtype='object')

In [10]:
train_data['History of HeartDisease or Attack'].value_counts()

,count
History of HeartDisease or Attack,
No,203322
Yes,18068


In [11]:
print(len(train_data))
train_data = train_data.dropna(subset=['History of HeartDisease or Attack'])
print(len(train_data))

223084
221390


In [ ]:
def create_new_features(df):
    train_data = df.copy()

    binary_map = {"Yes": 1, "No": 0}
    binary_cols = [
        "High Blood Pressure", "Told High Cholesterol", "Cholesterol Checked",
        "Smoked 100+ Cigarettes", "Diagnosed Stroke", "Diagnosed Diabetes",
        "Leisure Physical Activity", "Heavy Alcohol Consumption",
        "Health Care Coverage", "Doctor Visit Cost Barrier",
        "Difficulty Walking", "Fruit or Vegetable Intake (1+ per Day)"
    ]

    # 2. แปลงค่าเป็นตัวเลข
    for col in binary_cols:
        if col in train_data.columns:
            if train_data[col].dtype == object:
                train_data[col] = train_data[col].map(binary_map)

    # 3. คำนวณ Obesity Risk
    if 'Body Mass Index' in train_data.columns and 'Leisure Physical Activity' in train_data.columns:
        train_data['Obesity Risk'] = ((train_data['Body Mass Index'] >= 30) & (train_data['Leisure Physical Activity'] == 0)).astype(int)

    # 4. จัดการ BMI Category
    def categorize_bmi(bmi):
        if bmi < 18.5: return 0 # Underweight
        if bmi < 25: return 1   # Normal
        if bmi < 30: return 2   # Overweight
        return 3                # Obese

    if 'Body Mass Index' in train_data.columns:
        train_data['BMI_Cat'] = train_data['Body Mass Index'].apply(categorize_bmi)

    return train_data

In [ ]:
def create_new_features_test(df):
    train_data = df.copy()

    binary_map = {"Yes": 1, "No": 0}
    binary_cols = [
        "High Blood Pressure", "Told High Cholesterol", "Cholesterol Checked",
        "Smoked 100+ Cigarettes", "Diagnosed Stroke", "Diagnosed Diabetes",
        "Leisure Physical Activity", "Heavy Alcohol Consumption",
        "Health Care Coverage", "Doctor Visit Cost Barrier",
        "Difficulty Walking", "Fruit or Vegetable Intake (1+ per Day)"
    ]

    # 2. แปลงค่าเป็นตัวเลข
    for col in binary_cols:
        if col in train_data.columns:
            if train_data[col].dtype == object:
                train_data[col] = train_data[col].map(binary_map)

    # 3. คำนวณ Obesity Risk
    if 'Body Mass Index' in train_data.columns and 'Leisure Physical Activity' in train_data.columns:
        train_data['Obesity Risk'] = ((train_data['Body Mass Index'] >= 30) & (train_data['Leisure Physical Activity'] == 0)).astype(int)

    # 4. จัดการ BMI Category
    def categorize_bmi(bmi):
        if bmi < 18.5: return 0
        if bmi < 25: return 1
        if bmi < 30: return 2
        return 3

    if 'Body Mass Index' in train_data.columns:
        train_data['BMI_Cat'] = train_data['Body Mass Index'].apply(categorize_bmi) # เปลี่ยนกลับเป็น BMI_Cat

    return train_data

In [14]:
train_data = create_new_features(train_data)
train_data.head(3)

,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,...,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day),Obesity Risk,BMI_Cat
0,train_000001,No,1,1.0,1,40.68,1.0,0,0.0,0,...,0.0,Very Poor,1.0,Female,High school graduate,"$15,000 to less than $20,000",64,Yes,1,3
1,train_000002,No,0,0.0,0,24.36,1.0,0,0.0,1,...,1.0,Fair,0.0,Female,College graduate,"Less than $10,000",50,No,0,1
2,train_000003,No,1,1.0,1,27.33,0.0,0,0.0,0,...,1.0,Very Poor,1.0,Female,High school graduate,"$75,000 or more",61,Yes,0,2


In [15]:
df_yes = train_data[train_data['History of HeartDisease or Attack'] == 'Yes'].sample(n=18068, random_state=42)
df_no = train_data[train_data['History of HeartDisease or Attack'] == 'No'].sample(n=20000, random_state=42)

train_data = pd.concat([df_yes, df_no], ignore_index=True)

In [ ]:
save_path = 'best_model'
hyperparameters = {
    'GBM': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'CAT': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'XGB': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'RF': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],

}
time_limit=3600

In [17]:
predictor = TabularPredictor(label='History of HeartDisease or Attack',
                            problem_type='binary',
                            path=save_path,
                            eval_metric='f1'
                            ).fit(
                                train_data,
                                presets='best_quality',
                                num_stack_levels=1,
                                num_bag_folds=5,
                                time_limit=time_limit,
                                hyperparameters=hyperparameters
                            )
print("Finish!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.97 GB / 12.67 GB (86.6%)
Disk Space Avail:   64.66 GB / 112.64 GB (57.4%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a conseque

Finish!


In [23]:
test_data = create_new_features_test(test_data)

In [24]:
y_pred = predictor.predict_proba(test_data)
y_pred.head()

,No,Yes
0,0.517972,0.482028
1,0.679688,0.320312
2,0.318191,0.681809
3,0.836099,0.163901
4,0.849792,0.150208


In [25]:
y_pred['outcome'] = y_pred.apply(lambda row: 'Yes' if row['Yes'] > 0.505 else 'No', axis=1)
y_pred['outcome'].value_counts()

,count
outcome,
No,49947
Yes,24414


In [26]:
sample_submission = pd.read_csv("/content/sample_submission.csv")
sample_submission.head(2)

,ID,History of HeartDisease or Attack
0,test_000001,No
1,test_000002,No


In [27]:
sample_submission['History of HeartDisease or Attack'] = y_pred['outcome']
sample_submission.tail()

,ID,History of HeartDisease or Attack
74356,test_074357,No
74357,test_074358,Yes
74358,test_074359,Yes
74359,test_074360,Yes
74360,test_074361,No


In [28]:
sample_submission.to_csv("submission_heart_disease_prediction.csv", index=False)

In [29]:
from google.colab import files

files.download('submission_heart_disease_prediction.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>